FIRST ITERACTION

I will build a first version of an agent. First it will only copy and paste information from different Excel files.

In [1]:
%pip install pandas openpyxl

  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 KB 5.1 MB/s eta 0:00:0000:01
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.1/510.1 KB 34.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.3/349.3 KB 23.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


IMPORTS

In [1]:
import pandas as pd

print("Pandas is ready")

Pandas is ready


CREATING FILE TO CONSUME

In [13]:
base_razao_data = {
    "GRUPO": [
        "Operações",
        "Operações",
        "Tecnologia",
        "Marketing",
        "Operações",
        "Crédito"
    ],
    "SUB GRUPO": [
        "Cobrança",
        "Atendimento",
        "Infraestrutura",
        "Mídia",
        "BPO",
        "Bureau"
    ],
    "FORNECEDORES": [
        "ABC Collections",
        "Customer Care LTDA",
        "Google Cloud",
        "Meta",
        "Operations Partner",
        "Serasa"
    ],
    "EVENTO": [
        "Serviço de cobrança mensal",
        "Atendimento ao cliente",
        "Serviços cloud",
        "Campanha digital",
        "Suporte operacional",
        "Consulta de crédito"
    ],
    "CC DePara": [
        "CC_LOANS",
        "CC_LOANS",
        "CC_TECH",
        "CC_MARKETING",
        "CC_LOANS",
        "CC_CREDIT"
    ],
    "Resultado": [
        6700,
        9200,
        4300,
        15000,
        7800,
        8500
    ],
    "Forecast": [
        6500,
        9000,
        4500,
        14000,
        8000,
        8200
    ]
}

base_razao_df = pd.DataFrame(base_razao_data)

with pd.ExcelWriter("finance_actual_costs_realistic_dummy.xlsx", engine="openpyxl") as writer:
    base_razao_df.to_excel(writer, sheet_name="Base_Razão", index=False)

READING AND VALIDATE INPUT EXCEL

In [18]:
actual_costs_df = pd.read_excel("finance_actual_costs_realistic_dummy.xlsx")

## order by import columns to business validation
desired_cc = "CC_LOANS"
orderby_columns = ["GRUPO", "SUB GRUPO", "FORNECEDORES", "EVENTO"]

finance_validation_df = actual_costs_df[
    actual_costs_df["CC DePara"] == desired_cc
].sort_values(by=orderby_columns)


print("Actual costs file:")
finance_validation_df

Actual costs file:


,GRUPO,SUB GRUPO,FORNECEDORES,EVENTO,CC DePara,Resultado,Forecast
1,Operações,Atendimento,Customer Care LTDA,Atendimento ao cliente,CC_LOANS,9200,9000
4,Operações,BPO,Operations Partner,Suporte operacional,CC_LOANS,7800,8000
0,Operações,Cobrança,ABC Collections,Serviço de cobrança mensal,CC_LOANS,6700,6500


In [23]:
def read_excel(input_excel: str, desired_cc: str) -> pd.DataFrame:
    actual_costs_df = pd.read_excel(input_excel, sheet_name="Base_Razão")


    ## order by import columns to business validation
    orderby_columns = ["GRUPO", "SUB GRUPO", "FORNECEDORES", "EVENTO"]

    finance_validation_df = actual_costs_df[
        actual_costs_df["CC DePara"] == desired_cc
    ].sort_values(by=orderby_columns)

    return finance_validation_df

In [24]:
finance_validation_df = read_excel(
    "finance_actual_costs_realistic_dummy.xlsx",
    "CC_LOANS"
)

print("Actual costs file:")
finance_validation_df

Actual costs file:


,GRUPO,SUB GRUPO,FORNECEDORES,EVENTO,CC DePara,Resultado,Forecast
1,Operações,Atendimento,Customer Care LTDA,Atendimento ao cliente,CC_LOANS,9200,9000
4,Operações,BPO,Operations Partner,Suporte operacional,CC_LOANS,7800,8000
0,Operações,Cobrança,ABC Collections,Serviço de cobrança mensal,CC_LOANS,6700,6500


CREATE AND REGISTER THE OPEX DATA TO AN HISTORICAL EXCEL DATASET

In [19]:
import os

tracker_file_name = "opex_historical_tracker.xlsx"

os.path.exists(tracker_file_name)

False

In [3]:
actual_costs_df.to_excel("finance_actual_costs_may.xlsx", index=False)

CREATING FILE TO FILL

In [4]:
provision_template_data = {
    "month": [],
    "area": [],
    "cost_line": [],
    "supplier": [],
    "provision_amount": [],
    "justification": []
}

provision_template_df = pd.DataFrame(provision_template_data)

provision_template_df.to_excel("provision_template_may.xlsx", index=False)

In [5]:
actual_costs_df = pd.read_excel("finance_actual_costs_may.xlsx")
provision_template_df = pd.read_excel("provision_template_may.xlsx")

print("Actual costs file:")
display(actual_costs_df)

print("Provision template file:")
display(provision_template_df)

Actual costs file:


,month,area,cost_line,supplier,actual_amount,cost_center
0,May,Loans,CRM Tool,Salesforce,12000,CC_LOANS
1,May,Loans,Credit Bureau,Serasa,8500,CC_LOANS
2,May,Credit Card,Cloud Services,Google Cloud,4300,CC_CARD
3,May,Loans,Collection Agency,ABC Collections,6700,CC_LOANS
4,May,Marketing,Media Budget,Meta,15000,CC_MKT


Provision template file:


,month,area,cost_line,supplier,provision_amount,justification


The first interaction will be: read the file and display all costs for a specific cost center. This first agent action is to validate the information that the Finance team sent --- Before turning this into an agent tool, first make it work as normal Python.

In [6]:
cost_center_to_validate = "CC_LOANS"

filtered_costs_df = actual_costs_df[
    actual_costs_df["cost_center"] == cost_center_to_validate
]

total_amount = filtered_costs_df["actual_amount"].sum()

display(filtered_costs_df)

print(f"Total amount for {cost_center_to_validate}: {total_amount}")

,month,area,cost_line,supplier,actual_amount,cost_center
0,May,Loans,CRM Tool,Salesforce,12000,CC_LOANS
1,May,Loans,Credit Bureau,Serasa,8500,CC_LOANS
3,May,Loans,Collection Agency,ABC Collections,6700,CC_LOANS


Total amount for CC_LOANS: 27200


CREATING FUNCTION

In [ ]:
def validate_cost_center_costs(file_name: str, cost_center: str):
    actual_costs_df = pd.read_excel(file_name)

    filtered_costs_df = actual_costs_df[actual_costs_df["cost_center"] == cost_center]
    return filtered_costs_df

In [ ]:
validate_cost_center_costs(
    file_name="finance_actual_costs_may.xlsx", cost_center="CC_LOANS"
)

,month,area,cost_line,supplier,actual_amount,cost_center
0,May,Loans,CRM Tool,Salesforce,12000,CC_LOANS
1,May,Loans,Credit Bureau,Serasa,8500,CC_LOANS
3,May,Loans,Collection Agency,ABC Collections,6700,CC_LOANS
